##End-to-end MAG reconstruction

STEP 1. Genome Assembly

In [3]:
%%bash
wget -O reads.qza \
    https://polybox.ethz.ch/index.php/s/Yw34y55TDX98BA9/download

--2026-06-09 21:05:30--  https://polybox.ethz.ch/index.php/s/Yw34y55TDX98BA9/download


Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 548786775 (523M) [application/octet-stream]
Saving to: ‘reads.qza’

     0K .......... .......... .......... .......... ..........  0%  112K 80m0s
    50K .......... .......... .......... .......... ..........  0%  238K 58m47s
   100K .......... .......... .......... .......... ..........  0%  246K 51m18s
   150K .......... .......... .......... .......... ..........  0%  319K 45m29s
   200K .......... .......... .......... .......... ..........  0% 8٫59M 36m35s
   250K .......... .......... .......... .......... ..........  0%  207K 37m41s
   300K .......... .......... .......... .......... ..........  0%  124K 42m36s
   350K .......... .......... .......... .......... ..........  0% 29٫7M 37m19s
   400K .......... .......... .......... .......... ..........  0%  246K 37m12s
   450K ......

In [5]:
%%writefile parallel.config.toml
[parsl]

[[parsl.executors]]
class = "HighThroughputExecutor"
label = "default"

[parsl.executors.provider]
class = "LocalProvider"
max_blocks = 2

Writing parallel.config.toml


In [ ]:
%%bash
qiime assembly assemble-megahit \
    --i-reads reads.qza \
    --p-presets meta-sensitive \
    --p-num-cpu-threads 2 \
    --p-min-contig-len 500 \
    --o-contigs contigs.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[Contigs] to: contigs.qza


In [8]:
%%bash
wget -O reference-genomes.qza \
    https://polybox.ethz.ch/index.php/s/jA9FB8EF4YjP82x/download

--2026-06-09 23:19:12--  https://polybox.ethz.ch/index.php/s/jA9FB8EF4YjP82x/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11652707 (11M) [application/octet-stream]
Saving to: ‘reference-genomes.qza’

     0K .......... .......... .......... .......... ..........  0% 76٫2K 2m29s
    50K .......... .......... .......... .......... ..........  0%  244K 97s
   100K .......... .......... .......... .......... ..........  1%  344K 75s
   150K .......... .......... .......... .......... ..........  1% 7٫76M 57s
   200K .......... .......... .......... .......... ..........  2%  321K 52s
   250K .......... .......... .......... .......... ..........  2% 9٫98M 43s
   300K .......... .......... .......... .......... ..........  3%  348K 42s
   350K .......... .......... .......... .......... ..........  3% 7٫13M 36s
   400K .......... 

In [ ]:
%%bash
qiime assembly evaluate-quast \
    --i-contigs contigs.qza \
    --i-references reference-genomes.qza \
    --p-threads 4 \
    --p-min-contig 500 \
    --o-reference-genomes ref-genomes.qza \
    --o-results-table quast-results.qza \
    --o-visualization contigs-qc-quast.qzv \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metaquast.py -o /tmp/tmpt32kw19y/results --min-contig 500 --threads 4 --k-mer-size 101 --contig-thresholds 0,1000,5000,10000,25000,50000 --min-alignment 65 --min-identity 90.0 --ambiguity-usage one --ambiguity-score 0.99 /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample1_contigs.fa /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample2_contigs.fa /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample3_contigs.fa /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample4_contigs.fa -r /tmp/qiime2/sadia/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/B_subtilis.fasta -r /tmp/qiime2/sadia/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/E_coli_K-12.fasta -r /tmp/qiime2/sadia/data/654

STEP 2. MAG Binning
i.Contig indexing
ii.Read mapping
iii. Contig binning
iv. MAG quality control

In [ ]:
%%bash
qiime assembly index-contigs \
    --i-contigs contigs.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-index contigs-index.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[SingleBowtie2Index % Properties('contigs')] to: contigs-index.qza


In [ ]:
%%bash
qiime assembly map-reads \
    --i-index contigs-index.qza \
    --i-reads reads.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-alignment-maps reads-to-contigs-aln.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[AlignmentMap] to: reads-to-contigs-aln.qza


In [ ]:
%%bash
qiime annotate bin-contigs-metabat \
    --i-contigs contigs.qza \
    --i-alignment-maps reads-to-contigs-aln.qza \
    --p-num-threads 4 \
    --p-seed 100 \
    --o-mags mags.qza \
    --o-contig-map contig-map.qza \
    --o-unbinned-contigs unbinned-contigs.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools sort /tmp/qiime2/sadia/data/2f02f0b0-2ce7-49ca-bebc-8d2cc309d47c/data/sample1_alignment.bam -o /tmp/tmptjdnu3_t/sample1_alignment_sorted.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: jgi_summarize_bam_contig_depths --outputDepth /tmp/tmptjdnu3_t/sample1_depth.txt /tmp/tmptjdnu3_t/sample1_alignment_sorted.bam



Output depth matrix to /tmp/tmptjdnu3_t/sample1_depth.txt
jgi_summarize_bam_contig_depths 2.18 20250528_211447
Running with 8 threads to save memory you can reduce the number of threads with the OMP_NUM_THREADS variable
Output matrix to /tmp/tmptjdnu3_t/sample1_depth.txt
Opening all bam files and validating headers
Processing bam files with largest_contig=0
Thread 0 opening and reading the header for file: /tmp/tmptjdnu3_t/sample1_alignment_sorted.bam
Thread 0 opened the file: /tmp/tmptjdnu3_t/sample1_alignment_sorted.bam
Thread 0 processing bam 0: sample1_alignment_sorted.bam
Thread 0 finished reading bam 0: sample1_alignment_sorted.bam
Thread 0 finished: sample1_alignment_sorted.bam with 498734 reads and 280318 readsWellMapped (56.2059%)
Creating depth matrix file: /tmp/tmptjdnu3_t/sample1_depth.txt
Closing last bam file
Finished


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metabat2 -i /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample1_contigs.fa -a /tmp/tmptjdnu3_t/sample1_depth.txt -o /tmp/tmptjdnu3_t/sample1/bin --unbinned --numThreads 4 --seed 100

MetaBAT 2 (2.18) using minContig 2500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, maxEdges 200 and minClsSize 200000. with random seed=100
[00:00:00] Executing with 4 threads
[00:00:00] Parsing abundance file header [3.3Gb / 15.4Gb]
[00:00:00] Parsing assembly file [3.3Gb / 15.4Gb]
[00:00:00] Outputting contigs that are too short to /tmp/tmptjdnu3_t/sample1/bin.tooShort.fa
[00:00:00] Number of large contigs >= 2500 bp are500), 2526 short (>=1000) 95.0% (10055727 o 10583660:00:00     [3.3Gb / 15.4Gb]     000) 2.0% (212078 of 10583665), ETA 0:00:00

Output depth matrix to /tmp/tmpi1tum72p/sample2_depth.txt
jgi_summarize_bam_contig_depths 2.18 20250528_211447
Running with 8 threads to save memory you can reduce the number of threads with the OMP_NUM_THREADS variable
Output matrix to /tmp/tmpi1tum72p/sample2_depth.txt
Opening all bam files and validating headers
Processing bam files with largest_contig=0
Thread 0 opening and reading the header for file: /tmp/tmpi1tum72p/sample2_alignment_sorted.bam
Thread 0 opened the file: /tmp/tmpi1tum72p/sample2_alignment_sorted.bam
Thread 0 processing bam 0: sample2_alignment_sorted.bam
Thread 0 finished reading bam 0: sample2_alignment_sorted.bam
Thread 0 finished: sample2_alignment_sorted.bam with 1000000 reads and 855465 readsWellMapped (85.5465%)
Creating depth matrix file: /tmp/tmpi1tum72p/sample2_depth.txt
Closing last bam file
Finished


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metabat2 -i /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample2_contigs.fa -a /tmp/tmpi1tum72p/sample2_depth.txt -o /tmp/tmpi1tum72p/sample2/bin --unbinned --numThreads 4 --seed 100

MetaBAT 2 (2.18) using minContig 2500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, maxEdges 200 and minClsSize 200000. with random seed=100
[00:00:00] Executing with 4 threads
[00:00:00] Parsing abundance file header [3.5Gb / 15.4Gb]
[00:00:00] Parsing assembly file [3.5Gb / 15.4Gb]
[00:00:00] Outputting contigs that are too short to /tmp/tmpi1tum72p/sample2/bin.tooShort.fa
[00:00:00] ... processed 10152 seqs, 1737 long (>=2500), 3583 short (>=1000) 96.0% (19851245of 20676A 0:00:00     [3.5Gb / 15.4Gb]     =1000) 2.0% (416119 of 20676615), ETA 0:0

...

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metabat2 -i /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample3_contigs.fa -a /tmp/tmp6ef0pxpe/sample3_depth.txt -o /tmp/tmp6ef0pxpe/sample3/bin --unbinned --numThreads 4 --seed 100

MetaBAT 2 (2.18) using minContig 2500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, maxEdges 200 and minClsSize 200000. with random seed=100
[00:00:00] Executing with 4 threads
[00:00:00] Parsing abundance file header [3.9Gb / 15.4Gb]
[00:00:00] Parsing assembly file [3.9Gb / 15.4Gb]
[00:00:00] Outputting contigs that are too short to /tmp/tmp6ef0pxpe/sample3/bin.tooShort.fa
[00:00:00] Number of lar 6910 seqs, 1082 long (>=2500), 1718 short (>=1000) 95.1% (25508905 of 268090:00:00     [3.9Gb / 15.4Gb]     00) 2.1% (572427 of 26809438), ETA 0:00:00 

[bam_sort_core] merging from 2 files and 1 in-memory blocks...


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: jgi_summarize_bam_contig_depths --outputDepth /tmp/tmp2i15bacb/sample4_depth.txt /tmp/tmp2i15bacb/sample4_alignment_sorted.bam



Output depth matrix to /tmp/tmp2i15bacb/sample4_depth.txt
jgi_summarize_bam_contig_depths 2.18 20250528_211447
Running with 8 threads to save memory you can reduce the number of threads with the OMP_NUM_THREADS variable
Output matrix to /tmp/tmp2i15bacb/sample4_depth.txt
Opening all bam files and validating headers
Processing bam files with largest_contig=0
Thread 0 opening and reading the header for file: /tmp/tmp2i15bacb/sample4_alignment_sorted.bam
Thread 0 opened the file: /tmp/tmp2i15bacb/sample4_alignment_sorted.bam
Thread 0 processing bam 0: sample4_alignment_sorted.bam
Thread 0 finished reading bam 0: sample4_alignment_sorted.bam
Thread 0 finished: sample4_alignment_sorted.bam with 5000000 reads and 4918472 readsWellMapped (98.3694%)
Creating depth matrix file: /tmp/tmp2i15bacb/sample4_depth.txt
Closing last bam file
Finished


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metabat2 -i /tmp/qiime2/sadia/data/7a3531c6-846b-463b-a823-f098836c98d2/data/sample4_contigs.fa -a /tmp/tmp2i15bacb/sample4_depth.txt -o /tmp/tmp2i15bacb/sample4/bin --unbinned --numThreads 4 --seed 100

MetaBAT 2 (2.18) using minContig 2500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, maxEdges 200 and minClsSize 200000. with random seed=100
[00:00:00] Executing with 4 threads
[00:00:00] Parsing abundance file header [3.8Gb / 15.4Gb]
[00:00:00] Parsing assembly file [3.8Gb / 15.4Gb]
[00:00:00] Outputting contigs that are too short to /tmp/tmp2i15bacb/sample4/bin.tooShort.fa
[00:00:00] Number of large 37 seqs, 1265 long (>=2500), 361 short (>=1000) 96.0% (29516813 of 3074490:00:00     [3.8Gb / 15.4Gb]     00) 2.0% (619378 of 30744951), ETA 0:00:00 

In [ ]:
%%bash
qiime annotate fetch-busco-db \
    --p-lineages bacteria_odb12 \
    --o-db busco-db-bacteria.qza \
    --verbose


Fetching lineages: bacteria_odb12.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: busco --download_path /tmp/qiime2/sadia/processes/64914-1781030925.15@sadia/tmp/rachis-OutPath-6fyefww1 --download bacteria_odb12

2026-06-09 23:48:51 INFO:	Downloading information on latest versions of BUSCO data...
2026-06-09 23:49:00 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/lineages/bacteria_odb12.2026-05-22.tar.gz'
2026-06-09 23:52:35 INFO:	Decompressing file '/tmp/qiime2/sadia/processes/64914-1781030925.15@sadia/tmp/rachis-OutPath-6fyefww1/lineages/bacteria_odb12.tar.gz'
Download completed. 
Copying files from temporary directory to the final location...
Saved ReferenceDB[BUSCO] to: busco-db-bacteria.qza


In [ ]:
%%bash
qiime annotate evaluate-busco \
    --i-mags mags.qza \
    --i-db busco-db-bacteria.qza \
    --i-unbinned-contigs unbinned-contigs.qza \
    --p-lineage-dataset bacteria_odb12 \
    --p-cpu 2 \
    --o-results busco-results.qza \
    --o-visualization mags.qzv \
    --parallel-config parallel.config.toml \
    --verbose


Saved BUSCOResults to: busco-results.qza
Saved Visualization to: mags.qzv


In [ ]:
%%bash
qiime annotate filter-mags \
    --i-mags mags.qza \
    --m-metadata-file busco-results.qza \
    --p-where "completeness>50 AND contamination<10" \
    --p-on "mag" \
    --o-filtered-mags mags-filtered.qza \
    --verbose


/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing


Found 15 IDs to keep.
Saved SampleData[MAGs] to: mags-filtered.qza


STEP 3. MAG Dereplication

In [16]:
%%bash
qiime sourmash compute \
    --i-sequence-file mags-filtered.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature min-hash.qza \
    --verbose


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

** WARNING: the sourmash compute command is DEPRECATED as of 4.0 and
** will be removed in 5.0. Please see the 'sourmash sketch' command instead.

setting num_hashes to 0 because --scaled is set
computing signatures for files: /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/148b5aa1-e69f-4b20-b026-3874eb5545f9.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/3c8c0deb-fedb-4dea-95f6-d673f4bf7869.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/4b6e9c75-c335-4f4d-bd98-eba60255e450.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/5acc410b-8cef-434a-a75d-8e3b07b4da0e.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/61af53fd-2310-40fe-ad72-9037ca2e240d.fasta, /tmp/qiime2/sadia/processes/76749-17810

Saved MinHashSig to: min-hash.qza


In [17]:
%%bash
qiime sourmash compare \
    --i-min-hash-signature min-hash.qza \
    --p-ksize 105 \
    --o-compare-output min-hash-compare.qza \
    --verbose


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

loaded 15 si                                                                   6b2-814a-44e3-9bb6-3874eb5545f9.fasta.sig'6-d673f4bf7869.fasta.sig'asta.sig'8-eba60255e450.fasta.sig'asta.sig'loading '/tmp/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/5acc410b-8cef-434a-a75d-8e3b07b4da0e.fasta.sig'.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/61af53fd-2310-40fe-ad72-9037ca2e240d.fasta.sig'5-ca88286c94b9.fasta.sig'asta.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/913f57a8-39a4-41ad-a742-27b01737baf8.fasta.sig'<<<-43b1-b432-dbb2d225f158/data/913f57a8-39a4-41ad-a742-27b01737baf8.fasta.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/9d6f671d-b2d7-423b-8fd1-a838dbc7666e.fasta.sig'4-8a2b664d1491.fasta.sig'asta.sig'8-0f16532c40f9.fasta.sig'asta.sig'2-dbb2d225f158/data/aeacdfca-ae18-4c48-8678-0f16532c40f9.fasta.sig'2-ecb3

0-/tmp/qiime2/sad...	[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
1-/tmp/qiime2/sad...	[0.    1.    0.    0.619 0.    0.    0.    0.    0.918 0.    0.    0.
 0.001 0.    0.006]
2-/tmp/qiime2/sad...	[0.    0.    1.    0.    0.    0.    0.    0.998 0.    0.    0.    0.
 0.    0.86  0.   ]
3-/tmp/qiime2/sad...	[0.    0.619 0.    1.    0.027 0.    0.    0.    0.621 0.    0.    0.
 0.    0.    0.027]
4-/tmp/qiime2/sad...	[0.    0.    0.    0.027 1.    0.    0.    0.    0.004 0.    0.    0.
 0.    0.    0.884]
5-/tmp/qiime2/sad...	[0.    0.    0.    0.    0.    1.    0.985 0.    0.    0.    0.    0.
 0.443 0.    0.   ]
6-/tmp/qiime2/sad...	[0.    0.    0.    0.    0.    0.985 1.    0.    0.    0.    0.    0.
 0.44  0.    0.   ]
7-/tmp/qiime2/sad...	[0.    0.    0.998 0.    0.    0.    0.    1.    0.    0.    0.    0.
 0.    0.861 0.   ]
8-/tmp/qiime2/sad...	[0.    0.918 0.    0.621 0.004 0.    0.    0.    1.    0.    0.    0.
 0.001 0.    0.   ]
9-/tmp/qiime2/sad...	[0.    0.    0.    0.  

saving labels to: tmp.labels.txt
saving comparison matrix to: tmp


Saved DistanceMatrix to: min-hash-compare.qza


STEP 4. Taxonomic classification
i.Database preparation
ii.Read-based classification
iii. MAG-based classification

In [18]:
%%bash
qiime annotate dereplicate-mags \
    --i-mags mags-filtered.qza \
    --i-distance-matrix min-hash-compare.qza \
    --m-metadata-file busco-results.qza \
    --p-metadata-column completeness \
    --p-threshold 0.9 \
    --o-dereplicated-mags mags-derep.qza \
    --o-table table.qza \
    --verbose

/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing
/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/scipy/cluster/hierarchy.py:810: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  return linkage(y, method='ward', metric='euclidean')


Saved FeatureData[MAG] to: mags-derep.qza
Saved FeatureTable[PresenceAbsence] to: table.qza


In [ ]:
%%bash
qiime annotate build-kraken-db \
    --p-collection standard8 \
    --o-kraken2-db kraken2-db.qza \
    --o-bracken-db bracken-db.qza \
    --verbose


Found the latest "standard8" database: kraken/k2_standard_08gb_20250402.tar.gz.


Download finished. Extracting database files... Done.
Saved Kraken2DB to: kraken2-db.qza
Saved BrackenDB to: bracken-db.qza


In [3]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs reads.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-reads.qza \
    --o-outputs kraken2-hits-reads.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[Kraken2Report % Properties('reads')] to: kraken2-reports-reads.qza
Saved SampleData[Kraken2Output % Properties('reads')] to: kraken2-hits-reads.qza


In [4]:
%%bash
qiime annotate estimate-bracken \
    --i-kraken2-reports kraken2-reports-reads.qza \
    --i-db bracken-db.qza \
    --p-read-len 150 \
    --o-reports bracken-reports.qza \
    --o-taxonomy bracken-taxonomy.qza \
    --o-table bracken-table.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bracken -d /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1e-bf49-cc21e0082fd9/data -i /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt -o /tmp/tmp7zs48dur/sample3.bracken.output.txt -w /tmp/qiime2/sadia/processes/228314-1781094946.01@sadia/tmp/rachis-OutPath-luysp0hy/sample3.report.txt -t 0 -r 150 -l S

 >> Checking for Valid Options...
 >> Running Bracken 
      >> python src/est_abundance.py -i /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt -o /tmp/tmp7zs48dur/sample3.bracken.output.txt -k /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1e-bf49-cc21e0082fd9/data/database150mers.kmer_distrib -l S -t 0
PROGRAM START TIME: 06-10-2026 12:36:03


>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt


BRACKEN SUMMARY (Kraken report: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt)
    >>> Threshold: 0 
    >>> Number of species in sample: 453 
	  >> Number of species with reads > threshold: 453 
	  >> Number of species with reads < threshold: 0 
    >>> Total reads in sample: 1000000
	  >> Total reads kept at species level (reads > threshold): 164454
	  >> Total reads discarded (species reads < threshold): 0
	  >> Reads distributed: 794017
	  >> Reads not distributed (eg. no species above threshold): 8
	  >> Unclassified reads: 41521
BRACKEN OUTPUT PRODUCED: /tmp/tmp7zs48dur/sample3.bracken.output.txt
PROGRAM END TIME: 06-10-2026 12:36:03
  Bracken complete.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bracken -d /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1

>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample2.report.txt


BRACKEN SUMMARY (Kraken report: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample2.report.txt)
    >>> Threshold: 0 
    >>> Number of species in sample: 296 
	  >> Number of species with reads > threshold: 296 
	  >> Number of species with reads < threshold: 0 
    >>> Total reads in sample: 500000
	  >> Total reads kept at species level (reads > threshold): 82259
	  >> Total reads discarded (species reads < threshold): 0
	  >> Reads distributed: 397275
	  >> Reads not distributed (eg. no species above threshold): 7
	  >> Unclassified reads: 20459
BRACKEN OUTPUT PRODUCED: /tmp/tmp7zs48dur/sample2.bracken.output.txt
PROGRAM END TIME: 06-10-2026 12:36:04
  Bracken complete.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bracken -d /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1e-

...

BRACKEN SUMMARY (Kraken report: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample4.report.txt)
    >>> Threshold: 0 
    >>> Number of species in sample: 760 
	  >> Number of species with reads > threshold: 760 
	  >> Number of species with reads < threshold: 0 
    >>> Total reads in sample: 2500000
	  >> Total reads kept at species level (reads > threshold): 411187
	  >> Total reads discarded (species reads < threshold): 0
	  >> Reads distributed: 1985083
	  >> Reads not distributed (eg. no species above threshold): 29
	  >> Unclassified reads: 103701
BRACKEN OUTPUT PRODUCED: /tmp/tmp7zs48dur/sample4.bracken.output.txt
PROGRAM END TIME: 06-10-2026 12:36:04
  Bracken complete.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bracken -d /tmp/qiime2/sadia/data/dd6f4be9-424c-

>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample1.report.txt


BRACKEN SUMMARY (Kraken report: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample1.report.txt)
    >>> Threshold: 0 
    >>> Number of species in sample: 200 
	  >> Number of species with reads > threshold: 200 
	  >> Number of species with reads < threshold: 0 
    >>> Total reads in sample: 249367
	  >> Total reads kept at species level (reads > threshold): 41283
	  >> Total reads discarded (species reads < threshold): 0
	  >> Reads distributed: 197908
	  >> Reads not distributed (eg. no species above threshold): 4
	  >> Unclassified reads: 10172
BRACKEN OUTPUT PRODUCED: /tmp/tmp7zs48dur/sample1.bracken.output.txt
PROGRAM END TIME: 06-10-2026 12:36:05
  Bracken complete.


/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/q2_annotate/kraken2/select.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  table = pd.DataFrame(rows).fillna(False)


Saved SampleData[Kraken2Report % Properties('bracken')] to: bracken-reports.qza
Saved FeatureData[Taxonomy] to: bracken-taxonomy.qza
Saved FeatureTable[Frequency] to: bracken-table.qza


In [5]:
%%bash
qiime taxa barplot \
    --i-table bracken-table.qza \
    --i-taxonomy bracken-taxonomy.qza \
    --o-visualization bracken-barplot.qzv \
    --verbose


Saved Visualization to: bracken-barplot.qzv


In [6]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs mags-derep.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-mags.qza \
    --o-outputs kraken2-hits-mags.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved FeatureData[Kraken2Report % Properties('mags')] to: kraken2-reports-mags.qza
Saved FeatureData[Kraken2Output % Properties('mags')] to: kraken2-hits-mags.qza


In [7]:
%%bash
qiime annotate kraken2-to-mag-features \
    --i-reports kraken2-reports-mags.qza \
    --i-outputs kraken2-hits-mags.qza \
    --o-taxonomy mags-taxonomy.qza \
    --verbose


/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/q2_annotate/kraken2/select.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  table = pd.DataFrame(rows).fillna(False)


Saved FeatureData[Taxonomy] to: mags-taxonomy.qza


STEP 5. MAG abundance estimation
i.MAG indexing
ii.Read mapping
iii. MAG length estimation
iv.Abundance estimation
v.Taxonomic composition visualization

In [8]:
%%bash
qiime assembly index-derep-mags \
    --i-mags mags-derep.qza \
    --p-threads 4 \
    --p-seed 100 \
    --o-index mags-derep-index.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bowtie2-build --bmaxdivn 4 --dcv 1024 --offrate 5 --ftabchars 10 --threads 4 --seed 100 /tmp/tmpxbd1hoqo/merged.fasta /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index

Settings:
  Output files: "/tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 5 (one in 32)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 100
  Sizeofs: void*

Building a SMALL index


Reading reference sizes
  Time reading reference sizes: 00:00:00
Calculating joined length
Writing header
Reserving space for joined string
Joining reference sequences
  Time to join reference sequences: 00:00:00
bmax according to bmaxDivN setting: 7469841
Using parameters --bmax 5602381 --dcv 1024
  Doing ahead-of-time memory usage test
  Passed!  Constructing with these parameters: --bmax 5602381 --dcv 1024
Constructing suffix-array element generator
Building DifferenceCoverSample
  Building sPrime
  Building sPrimeOrder
  V-Sorting samples
  V-Sorting samples time: 00:00:01
  Allocating rank array
  Ranking v-sort output
  Ranking v-sort output time: 00:00:00
  Invoking Larsson-Sadakane on ranks
  Invoking Larsson-Sadakane on ranks time: 00:00:00
  Sanity-checking and returning
Building samples
Reserving space for 12 sample suffixes
Generating random suffixes
QSorting 12 sample offsets, eliminating duplicates
QSorting sample offsets, eliminating duplicates time: 00:00:00
Multikey QS

Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.3.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.3.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.4.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.4.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.1.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.1.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.2.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.2.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.rev.1.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/ra

Saved FeatureData[SingleBowtie2Index % Properties('mags')] to: mags-derep-index.qza


In [9]:
%%bash
qiime assembly map-reads \
    --i-index mags-derep-index.qza \
    --i-reads reads.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-alignment-maps reads-to-mags-aln.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved FeatureData[AlignmentMap] to: reads-to-mags-aln.qza


In [10]:
%%bash
qiime annotate get-feature-lengths \
    --i-features mags-derep.qza \
    --o-lengths mags-derep-lengths.qza \
    --verbose


Saved FeatureData[SequenceCharacteristics % Properties('length')] to: mags-derep-lengths.qza


In [11]:
%%bash
qiime annotate estimate-abundance \
    --i-alignment-maps reads-to-mags-aln.qza \
    --i-feature-lengths mags-derep-lengths.qza \
    --p-metric tpm \
    --p-min-mapq 42 \
    --p-threads 4 \
    --o-abundances mags-abundances.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools view -q 42 -m 0 --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample1_alignment.bam | samtools sort -o /tmp/tmpunrzf5rx/sample1.bam --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample1_alignment.bam



[bam_sort_core] merging from 0 files and 4 in-memory blocks...


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools coverage -Q 0 -l 0 -o /tmp/tmpunrzf5rx/sample1.coverage.tsv /tmp/tmpunrzf5rx/sample1.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools view -q 42 -m 0 --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample2_alignment.bam | samtools sort -o /tmp/tmpunrzf5rx/sample2.bam --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample2_alignment.bam



[bam_sort_core] merging from 0 files and 4 in-memory blocks...


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools coverage -Q 0 -l 0 -o /tmp/tmpunrzf5rx/sample2.coverage.tsv /tmp/tmpunrzf5rx/sample2.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools view -q 42 -m 0 --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample4_alignment.bam | samtools sort -o /tmp/tmpunrzf5rx/sample4.bam --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample4_alignment.bam



[bam_sort_core] merging from 0 files and 4 in-memory blocks...


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools coverage -Q 0 -l 0 -o /tmp/tmpunrzf5rx/sample4.coverage.tsv /tmp/tmpunrzf5rx/sample4.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools view -q 42 -m 0 --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample3_alignment.bam | samtools sort -o /tmp/tmpunrzf5rx/sample3.bam --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample3_alignment.bam



[bam_sort_core] merging from 0 files and 4 in-memory blocks...


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools coverage -Q 0 -l 0 -o /tmp/tmpunrzf5rx/sample3.coverage.tsv /tmp/tmpunrzf5rx/sample3.bam

Saved FeatureTable[Frequency % Properties('tpm')] to: mags-abundances.qza


In [12]:
%%bash
qiime taxa barplot \
    --i-table mags-abundances.qza \
    --i-taxonomy mags-taxonomy.qza \
    --o-visualization mags-taxa-barplot.qzv \
    --verbose


Saved Visualization to: mags-taxa-barplot.qzv
